In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

# Load dataset
df = pd.read_csv("Pahul_dataset_segmented_full.csv")

# Column names
group_col = "id"
label_col = "disease_y"

# -----------------------------
# 1. Create patient-level table
# -----------------------------
# One row per patient, with that patient's disease label
patient_df = df.groupby(group_col)[label_col].nunique().reset_index(name="n_labels")

# Check that each patient has only one disease label
if (patient_df["n_labels"] > 1).any():
    bad_ids = patient_df.loc[patient_df["n_labels"] > 1, group_col].tolist()
    raise ValueError(f"Some patients have multiple disease labels: {bad_ids[:10]}")

patient_labels = df.groupby(group_col)[label_col].first().reset_index()

# -----------------------------
# 2. Stratified split by patient
# -----------------------------
# First split: train (70% patients) and temp (30% patients)
train_patients, temp_patients = train_test_split(
    patient_labels,
    test_size=0.30,
    stratify=patient_labels[label_col],
    random_state=42
)

# Second split: validation (15% patients) and test (15% patients)
val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    stratify=temp_patients[label_col],
    random_state=42
)

# -----------------------------
# 3. Map patient splits back to full segmented dataset
# -----------------------------
train_ids = set(train_patients[group_col])
val_ids = set(val_patients[group_col])
test_ids = set(test_patients[group_col])

train_df = df[df[group_col].isin(train_ids)].copy()
val_df = df[df[group_col].isin(val_ids)].copy()
test_df = df[df[group_col].isin(test_ids)].copy()

# -----------------------------
# 4. Save CSVs
# -----------------------------
train_df.to_csv("train_group_stratified.csv", index=False)
val_df.to_csv("val_group_stratified.csv", index=False)
test_df.to_csv("test_group_stratified.csv", index=False)

# -----------------------------
# 5. Checks
# -----------------------------
print("ROW COUNTS")
print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))

print("\nPATIENT COUNTS")
print("Train patients:", train_df[group_col].nunique())
print("Val patients:", val_df[group_col].nunique())
print("Test patients:", test_df[group_col].nunique())

train_ids_check = set(train_df[group_col].unique())
val_ids_check = set(val_df[group_col].unique())
test_ids_check = set(test_df[group_col].unique())

print("\nPATIENT OVERLAP CHECK")
print("Train-Val overlap:", len(train_ids_check & val_ids_check))
print("Train-Test overlap:", len(train_ids_check & test_ids_check))
print("Val-Test overlap:", len(val_ids_check & test_ids_check))

print("\nDISEASE DISTRIBUTION BY PATIENT")
print("Train:\n", train_patients[label_col].value_counts(normalize=True))
print("\nVal:\n", val_patients[label_col].value_counts(normalize=True))
print("\nTest:\n", test_patients[label_col].value_counts(normalize=True))

print("\nDISEASE DISTRIBUTION BY SEGMENT")
print("Train:\n", train_df[label_col].value_counts(normalize=True))
print("\nVal:\n", val_df[label_col].value_counts(normalize=True))
print("\nTest:\n", test_df[label_col].value_counts(normalize=True))

ROW COUNTS
Train rows: 9644
Val rows: 1929
Test rows: 2161

PATIENT COUNTS
Train patients: 1759
Val patients: 377
Test patients: 378

PATIENT OVERLAP CHECK
Train-Val overlap: 0
Train-Test overlap: 0
Val-Test overlap: 0

DISEASE DISTRIBUTION BY PATIENT
Train:
 disease_y
healthy    0.612280
covid      0.192723
asthma     0.131325
copd       0.063673
Name: proportion, dtype: float64

Val:
 disease_y
healthy    0.612732
covid      0.190981
asthma     0.132626
copd       0.063660
Name: proportion, dtype: float64

Test:
 disease_y
healthy    0.611111
covid      0.193122
asthma     0.132275
copd       0.063492
Name: proportion, dtype: float64

DISEASE DISTRIBUTION BY SEGMENT
Train:
 disease_y
healthy    0.492845
asthma     0.245852
copd       0.148279
covid      0.113024
Name: proportion, dtype: float64

Val:
 disease_y
healthy    0.481078
asthma     0.244686
copd       0.174702
covid      0.099533
Name: proportion, dtype: float64

Test:
 disease_y
healthy    0.502082
asthma     0.243406
copd